# Evaluation — CS 5542 Challenge 1

Evaluation of **method quality**, distinct from the 418 AC-derived tests, which verify **spec compliance**.

Every table is written to `notebooks/results/` and every figure to `notebooks/figures/`, so the report
quotes measured numbers rather than restating claims.

| Section | Acceptance criterion |
|---|---|
| 1 · Retrieval comparison | AC-13.1 |
| 2 · End-to-end results per profile | AC-13.2 |
| 3 · Latency | AC-13.3 |
| 4 · Scalability | AC-13.4 |
| 5 · Calibration evidence | AC-13.5 |
| 6 · Component sensitivity | AC-13.6 |

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, plotly.express as px, plotly.io as pio
from pathlib import Path
import sys; sys.path.insert(0, "..")

from src.evaluation import (FIGURES, RESULTS, RELEVANCE_PROXY_NOTE, calibration_evidence,
                            component_sensitivity, latency_profile, pooled_candidates,
                            precision_at_k, recall_of_top_scored, scalability,
                            semantic_correlation, sensitivity_all_profiles)
from src.filters import warm_caches
from src.indexing import JobIndex
from src.personal_kb import PersonalKB
from src.pipeline import search
from src.profiles import PRESETS

FIGURES.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)
warm_caches()
ROOT = Path("..").resolve()
jobs = pd.read_parquet(ROOT / "data/processed/jobs_tech.parquet")
index = JobIndex.load_or_build(jobs, index_dir=ROOT / "data/index")
kbs = {k: PersonalKB.build(p.resume_text, p.career_goals) for k, p in PRESETS.items()}
print(f"corpus {len(jobs):,} jobs · {len(PRESETS)} profiles")

## 1 · Retrieval comparison (AC-13.1 v1.1)

Run on the **filter survivors**, asking the question the retrieval stage exists to answer:
*does it surface the jobs that full scoring would rank highest?*

Score every survivor, take the true top-20 by final match score, then measure each method's
**recall@k** against that set. No human labels, and not circular — retrieval ranks by BM25 and
embedding similarity, scoring ranks by eight weighted components.

> **Why this replaced the original metric.** A title-family relevance proxy returned precision@10
> of **1.00 for both BM25 and hybrid on every profile** — it could not separate the methods at all.
> That was a limitation of the metric, not evidence the methods were equivalent. The proxy and the
> pooled-judgment pools are retained below as a cross-check.

In [ ]:
frames = []
for key, p in PRESETS.items():
    r = recall_of_top_scored(jobs, index, p, kbs[key])
    r.insert(0, "profile", p.name)
    frames.append(r)
recall = pd.concat(frames, ignore_index=True)
recall.to_csv(RESULTS / "recall_of_top_scored.csv", index=False)
recall.pivot_table(index=["profile", "k"], columns="mode", values="recall")

In [ ]:
fig = px.line(recall, x="k", y="recall", color="mode", facet_col="profile", markers=True,
              range_y=[0, 1.02], title="Recall of the true top-20 scored jobs")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1][:28]))
fig.write_image(FIGURES / "retrieval_comparison.png", scale=2); fig

**Hybrid is the most robust retriever.** At k=200 it recovers 0.95 / 1.00 / 0.95 of the true top-20
across the three profiles, matching or beating both single retrievers everywhere. BM25 alone is
weakest (0.55 on the data profile); dense alone is close but loses to hybrid on the backend profile.
**Neither single retriever wins everywhere**, which is the asymmetry hybrid exists to absorb.

Extending the sweep is what set the production `k`: **hybrid reaches 100% recall at k=400**, while
BM25 alone needs k=1400. That measurement reversed an earlier decision (D13 v1), which had skipped
retrieval below a survivor threshold — solving a `k`-sizing problem by deleting the stage, and
leaving this very comparison evaluating something the application bypassed. Production now runs
`k = max(400, 25% of survivors)`.

**Cross-encoder reranking (REQ-8) is implemented but disabled.** At k=400 retrieval already recovers
everything scoring wants, so a reranker has nothing left to recover and would add seconds to a
sub-second search. It is kept, tested and reported rather than asserted away.

## 2 · End-to-end results per profile (AC-13.2)

In [ ]:
rows = []
for key, p in PRESETS.items():
    res = search(jobs, index, p, kbs[key])
    for rank, r in enumerate(res.results, 1):
        rows.append({"profile": p.name, "rank": rank, "score": r["score"], "tier": r["tier"],
                     "title": r["job"]["title"][:46], "location": r["job"]["location_raw"],
                     "matched": len(r["matched_skills"]), "missing": len(r["missing_skills"])})
top5 = pd.DataFrame(rows); top5.to_csv(RESULTS / "top5_by_profile.csv", index=False)
top5

## 3 · Latency (AC-13.3)\n\nMedian and p95 per stage over 20 runs.

In [ ]:
lat = latency_profile(jobs, index, PRESETS["data_science_student"], kbs["data_science_student"], runs=20)
lat.to_csv(RESULTS / "latency.csv", index=False)
fig = px.bar(lat[lat.stage != "total"], x="stage", y="median_ms",
             error_y=lat[lat.stage != "total"]["p95_ms"] - lat[lat.stage != "total"]["median_ms"],
             title="Per-stage latency (median, whisker to p95)")
fig.write_image(FIGURES / "latency.png", scale=2); display(lat); fig

**Filtering before retrieval costs ~62 ms** and removes 90–99% of the corpus before the expensive
stages touch it. The draft plan called stage ordering *"a performance tuning decision, not an
architectural one"* and filtered *after* retrieving 200 candidates; at the measured survival rates
that ordering leaves roughly 19, 12 and 2 candidates to rank. It is architectural, and the
performance argument runs the other way too.

Retrieval is now the largest stage (365 ms) because it always runs and searches a larger `k`. That is
the cost of the recall guarantee above, stated rather than hidden — a total of **556 ms median /
576 ms p95** is comfortably interactive.

## 4 · Scalability (AC-13.4)

In [ ]:
sc = scalability()
sc.to_csv(RESULTS / "scalability.csv", index=False)
fig = px.line(sc, x="rows", y="seconds", markers=True,
              title="Ingestion wall-clock vs corpus size (DuckDB, single machine)")
fig.write_image(FIGURES / "scalability.png", scale=2); display(sc); fig

Throughput **rises** with corpus size (7k → 37.6k rows/sec) because fixed startup cost dominates the
small runs. On the 785,741-row `data_jobs` corpus a `GROUP BY` with a median aggregate returns in
0.02 s — roughly 40 M rows/sec.

That is the evidence behind **D9**: at this scale the data fits single-machine, so Spark's JVM startup
and shuffle overhead would be cost without benefit. The crossover is documented rather than asserted.

## 5 · Calibration evidence (AC-13.5)

In [ ]:
cal = calibration_evidence(index, PRESETS["data_science_student"])
cal.to_csv(RESULTS / "calibration_evidence.csv", index=False)
fig = px.histogram(cal, x="raw_cosine", color="population", barmode="overlay", nbins=60,
                   title="Before — raw cosine similarity")
fig.write_image(FIGURES / "calibration_raw.png", scale=2); fig

In [ ]:
fig = px.histogram(cal, x="calibrated", color="population", barmode="overlay", nbins=60,
                   title="After — calibrated against a fixed background")
fig.write_image(FIGURES / "calibration_after.png", scale=2); fig

Raw cosine clusters in a narrow band, so the semantic components would barely vary between jobs.
Calibration spreads the **retrieved candidates** across the full range while compressing the rest of
the corpus toward zero — which is correct: those jobs are irrelevant.

This is the principled answer to the Stage 2 AI code's `sim * 140.0` rescaling
(`agent-exercise:src/matcher.py:117`, whose comment admits the constant was chosen so that
"strong matches reach 85-95%"). Same goal; constants measured from the corpus instead of picked.

## 6 · Component sensitivity (AC-13.6)\n\nZero each component in turn and see whether the top 5 moves.

In [ ]:
detail, summary = sensitivity_all_profiles(jobs, index, PRESETS, kbs)
detail.to_csv(RESULTS / "component_sensitivity_all.csv", index=False)
summary.to_csv(RESULTS / "component_sensitivity_summary.csv", index=False)
fig = px.bar(summary, x="component", y="mean_jobs_replaced",
             title="Top-5 jobs replaced when a component is zeroed (mean over 3 profiles)")
fig.write_image(FIGURES / "component_sensitivity.png", scale=2); display(summary); fig

In [ ]:
corr = pd.DataFrame([{"profile": PRESETS[k].name,
                      "correlation": round(semantic_correlation(jobs, index, PRESETS[k], kbs[k]), 3)}
                     for k in PRESETS])
corr.to_csv(RESULTS / "semantic_correlation.csv", index=False); corr

**Measured across all three profiles**, because a single profile's result is not a property of the
weight.

| Component | Weight | Mean jobs replaced | Profiles affected |
|---|---:|---:|---:|
| Required skills | 30% | 3.33 | 3/3 |
| Experience | 15% | 1.33 | 2/3 |
| **Education** | **4%** | **0.67** | **3/3** |
| Career goals ~ description | 13% | 0.67 | 3/3 |
| Title | 5% | 0.67 | 2/3 |
| Résumé evidence ~ description | 17% | 0.67 | 2/3 |
| Location | 8% | 0.33 | 2/3 |
| Preferred skills | 8% | 0.00 | 1/3 |

**Q6 closes — keep the weight, discard the reasoning.** Education at 4% affects all three profiles.
Its original justification — that most postings state no requirement, so the component sits at
neutral and cannot discriminate — was refuted twice: first by 58.5% coverage, then by this.

**Q7: preferred skills (8%) is the weakest component**, affecting one profile and replacing no top-5
job on average. The explanation is in the data: only **7.0%** of postings list any preferred skills,
and AC-9.4 renormalises the weight away on the other 93%. Location (8%) is low but **not inert** — it
affects two of three profiles. An earlier reading called it inert, which was an artifact of testing a
single profile whose top results are mostly remote: *a conclusion about a weight needs agreement
across profiles.*

Recorded as a measured limitation rather than re-tuned without time to re-validate every acceptance
criterion; the weights also trace directly to the Stage 1 and Stage 2 documents, and that
traceability is worth more to the report than a marginal ranking gain.

**Q2 closes** via the correlation below: **0.38, −0.08, 0.60** — related, but not the same signal.
Before AC-9.10 excluded the career-goals chunk from the evidence max they were *identical* on most
results, which meant 30% of the weight was one signal counted twice.